<div style="font-size: 0.85em; line-height: 1.5;">

<h3>Self-Query Retriever</h3>

<p><strong>What is it?</strong><br>
A retriever that uses an LLM to automatically generate a <em>semantic query</em> and a <em>metadata filter</em> from the user’s question. It then applies that filter to the vector store to retrieve only the most relevant documents.</p>

<p><strong>How it works</strong></p>
<ol>
  <li>User asks a question, e.g., <em>“What are the common diseases in Nigeria according to the health document?”</em></li>
  <li>The LLM parses the question into two parts:
    <ul>
      <li><strong>Semantic query:</strong> “common diseases in Nigeria”</li>
      <li><strong>Metadata filter:</strong> <code>source contains "nigeria_health"</code></li>
    </ul>
  </li>
  <li>The vector store applies the filter while searching, returning only chunks from the health PDF.</li>
  <li>The LLM answers using that focused context.</li>
</ol>

<p><strong>Visualisation</strong></p>
<pre>
User Query
   │
   ▼
[LLM parses query]
   ├── Semantic query: "common diseases in Nigeria"
   └── Metadata filter: source contains "nigeria_health"
   │
   ▼
[Vector Store with filter] → Relevant chunks only
   │
   ▼
[LLM Generator] → Answer
</pre>

<p><strong>Why use it?</strong></p>
<ul>
  <li><strong>Improves precision</strong> – avoids irrelevant chunks from other documents.</li>
  <li><strong>No manual filtering</strong> – the LLM infers the right filter automatically.</li>
  <li><strong>Handles complex queries</strong> – can combine multiple metadata fields.</li>
  <li><strong>Production-ready</strong> – especially useful when your knowledge base has multiple document types.</li>
</ul>

<p><strong>Implementation</strong><br>
We will use LangChain’s <code>SelfQueryRetriever</code> with a vector store and metadata field information.</p>

</div>

Step 1: Imports

In [1]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader, BSHTMLLoader
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

Step 2: Load Relevant Documents and Create Chunks

In [3]:
# Load only the health and agriculture documents (same as clean setup)
health_pdf = PyPDFLoader('../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf').load()
crop_pdf = PyPDFLoader('../../04_data_ingestion_document_processing/data/crop_disease.pdf').load()
agri_html = BSHTMLLoader('../../04_data_ingestion_document_processing/data/agriculture.html', open_encoding='utf-8', bs_kwargs={'features': 'html.parser'}).load()
agri_txt = TextLoader('../../04_data_ingestion_document_processing/data/agriculture.txt', encoding='utf-8').load()

In [4]:
# Combine all documents
all_docs = health_pdf + crop_pdf + agri_html + agri_txt
print(f'Loaded {len(all_docs)} documents.')

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

chunks = splitter.split_documents(all_docs)

# Add metadata
for i, chunk in enumerate(chunks):
    source = chunk.metadata.get('source', '')
    file_name = source.split('\\')[-1] if '\\' in source else source.split('/')[-1]
    chunk.metadata['file_name'] = file_name
    chunk.metadata['doc_type'] = (
        'pdf' if file_name.endswith('.pdf')
        else 'html' if file_name.endswith('.html')
        else 'txt'
    )
    chunk.metadata['language'] = 'English'
    chunk.metadata['chunk_id'] = f'{file_name}_{i+1:03d}'

print(f'Created {len(chunks)} chunks.')

    

Loaded 37 documents.
Created 185 chunks.


Step 3: Create Vector Store and Base Retriever

In [5]:
# Create embedding model
embeddings = OpenAIEmbeddings()

# Build in-memory Chroma vector store from chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=None
)

print('Vector store ready.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store ready.


Step 4: Define Metadata Field Info

In [6]:
metadata_field_info = [
    AttributeInfo(
        name='source',
        description='The file path of the document. Contains keywords like "nigeria_health" or "crop_disease" or "agriculture".',
        type='string',
    ),
    AttributeInfo(
        name='doc_type',
        description='The type of document: pdf, html, txt',
        type='string',
    ),
    AttributeInfo(
        name='language',
        description='Language of the document, e.g., English',
        type='string',
    ),
]

Step 5: Create the SelfQueryRetriever

In [7]:
# LLM for parsing queries (same as before)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Document content description
document_content_description = 'Documents about agriculture, public health, and diseases in Nigeria'

# Create self-query retriever
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True    # shows the generated query and filter
)

print('SelfQueryRetriever ready.')

SelfQueryRetriever ready.


Step 6: Test the SelfQueryRetriever

In [8]:
# Ask a question that should retrieve only from health documents
question = 'What are the common diseases in Nigeria according to the health document?'

# Retrieve using self-query retriever
docs = self_query_retriever.invoke(question)

print(f'Retrieved {len(docs)} chunks:\n')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:200]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print(f'   Type: {doc.metadata.get("doc_type", "unknown")}')

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved 4 chunks:

1. like Hypertension, Cancer, Obesity etc. Most of the 
Nigerians (young and old) die of different             
preventable diseases such as HIV/AIDS, tuberculo-
sis, malaria, vaccine preventable disease
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
   Type: pdf
2. include cardiovascular disease, cancer, diabetes, chronic 
respiratory diseases, sickle cell disease, asthma, coronary 
heart disease, obesity, stroke, hypertension, road traffic 
injuries and mental 
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
   Type: pdf
3. major health problem in Nigeria.  
 The top causes of death in Nigeria are; malaria, 
lower respiratory infections, HIV/AIDS,      
diarrheal diseases, road injuries, protein -energy 
malnutrition, c
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
   Type: pdf
4. diseases, alcoho

Agriculture / Crop Disease Filter

In [9]:
# Ask a question that should retrieve only from health documents
question = 'What are the common crop diseases and their control methods?'

# Retrieve using self-query retriever
docs = self_query_retriever.invoke(question)

print(f'Retrieved {len(docs)} chunks:\n')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:200]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print(f'   Type: {doc.metadata.get("doc_type", "unknown")}')

Retrieved 4 chunks:

1. Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cutti
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
   Type: html
2. Common Crop Diseases and Control

1. Cassava Mosaic Disease
   Affected crop: Cassava
   Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
   Control: Use disease-free cutting
   Source: ../../04_data_ingestion_document_processing/data/agriculture.txt
   Type: txt
3. Management: 
 Use of resistant varieties. 
 Practice crop rotation. Cut out 
infected parts if only few.
   Source: ../../04_data_ingestion_document_processing/data/crop_disease.pdf
   Type: pdf
4. Fisheries and aquaculture provide employment and protein for many families. Catfish and tilapia are commonly farmed in ponds and tanks.

Common Crop Diseases and Control
   Source: ../../0

In [10]:
# Ask a question that should retrieve only from health documents
question = 'What are the epidemic prone diseases in Nigeria from the English health PDF?'

# Retrieve using self-query retriever
docs = self_query_retriever.invoke(question)

print(f'Retrieved {len(docs)} chunks:\n')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:200]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print(f'   Type: {doc.metadata.get("doc_type", "unknown")}')

Retrieved 4 chunks:

1. like Hypertension, Cancer, Obesity etc. Most of the 
Nigerians (young and old) die of different             
preventable diseases such as HIV/AIDS, tuberculo-
sis, malaria, vaccine preventable disease
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
   Type: pdf
2. major health problem in Nigeria.  
 The top causes of death in Nigeria are; malaria, 
lower respiratory infections, HIV/AIDS,      
diarrheal diseases, road injuries, protein -energy 
malnutrition, c
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
   Type: pdf
3. include cardiovascular disease, cancer, diabetes, chronic 
respiratory diseases, sickle cell disease, asthma, coronary 
heart disease, obesity, stroke, hypertension, road traffic 
injuries and mental 
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
   Type: pdf
4. programs designe

In [11]:
# Ask a question that should retrieve only from health documents
question = 'What are the symptoms of maize smut in the HTML document?'

# Retrieve using self-query retriever
docs = self_query_retriever.invoke(question)

print(f'Retrieved {len(docs)} chunks:\n')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:200]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print(f'   Type: {doc.metadata.get("doc_type", "unknown")}')

Retrieved 4 chunks:

1. 2. Maize Smut
Affected Crop: Maize
Symptoms: Large grey or black galls on ears, stalks, and leaves.
Control/Cure: Remove and destroy infected plants, rotate crops, and plant resistant hybrids.
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
   Type: html
2. 3. Rice Blast
Affected Crop: Rice
Symptoms: Spindle-shaped lesions on leaves, panicle damage, and reduced grain quality.
Control/Cure: Use resistant varieties, avoid excessive nitrogen fertiliser, and
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
   Type: html
3. Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cutti
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
   Type: html
4. 4. Tomato Leaf Curl Virus
Affected Crop: Tomatoes and peppers
Symptoms: Curling and yellowing of le